In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')

print("Libraries imported successfully!")

Libraries imported successfully!


In [19]:
# Load the main orders dataset
df = pd.read_csv('../data/raw/ecommerce_orders_synthetic.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head(10)

Dataset shape: (50000, 18)

Columns: ['order_id', 'order_date', 'category', 'order_value', 'city', 'city_tier', 'warehouse', 'payment_method', 'dispatch_date', 'delivery_date', 'delivery_status', 'is_returned', 'return_initiated_date', 'return_reason', 'refund_completed_date', 'refund_status', 'refund_delay_days', 'sla_breach']


,order_id,order_date,category,order_value,city,city_tier,warehouse,payment_method,dispatch_date,delivery_date,delivery_status,is_returned,return_initiated_date,return_reason,refund_completed_date,refund_status,refund_delay_days,sla_breach
0,ORD770487,2024-10-31,Bottoms,1925,Nashik,Tier3,WH_Delhi,COD,2024-11-01,2024-11-07,Delivered,False,NaN,NaN,NaN,NaN,NaN,NaN
1,ORD671858,2024-10-25,Dresses,2727,Hyderabad,Metro,WH_Hyderabad,Prepaid,2024-10-26,2024-10-30,Delivered,True,2024-10-31,Size/Fit Issue,2024-11-06,Completed,6.0,False
2,ORD850800,2025-03-18,Tops,1615,Mysore,Tier3,WH_Kolkata,COD,2025-03-19,2025-03-27,Delivered,False,NaN,NaN,NaN,NaN,NaN,NaN
3,ORD106814,2024-11-12,Tops,1364,Pune,Metro,WH_Bangalore,Prepaid,2024-11-14,2024-11-19,Delivered,True,2024-11-26,Size/Fit Issue,2024-12-24,Pending,28.0,True
4,ORD988662,2024-12-30,Tops,1040,Coimbatore,Tier2,WH_Mumbai,Prepaid,2025-01-02,2025-01-06,Delivered,False,NaN,NaN,NaN,NaN,NaN,NaN
5,ORD496922,2024-10-23,Tops,1099,Hyderabad,Metro,WH_Hyderabad,Prepaid,2024-10-25,2024-10-28,Delivered,True,2024-11-01,Quality/Damage,2024-11-06,Completed,5.0,False
6,ORD183667,2024-12-01,Tops,705,Visakhapatnam,Tier2,WH_Kolkata,Prepaid,2024-12-03,2024-12-09,Delivered,True,2024-12-15,Size/Fit Issue,2024-12-16,Completed,1.0,False
7,ORD379946,2025-03-31,Accessories,445,Kochi,Tier2,WH_Hyderabad,Prepaid,2025-04-03,2025-04-12,Delivered,False,NaN,NaN,NaN,NaN,NaN,NaN
8,ORD584714,2025-01-08,Tops,1809,Nagpur,Tier2,WH_Hyderabad,Prepaid,2025-01-09,2025-01-13,Delivered,True,2025-01-14,Size/Fit Issue,2025-01-17,Completed,3.0,False
9,ORD520651,2024-12-10,Tops,931,Jaipur,Tier2,WH_Hyderabad,Prepaid,2024-12-13,2024-12-22,Delivered,False,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
# Load complaints dataset
df_complaints = pd.read_csv('../data/raw/complaints_aggregated.csv')
print(f"Complaints dataset shape: {df_complaints.shape}")
df_complaints.head()

Complaints dataset shape: (300, 6)


,complaint_id,complaint_date,complaint_type,complaint_text,sentiment_score,escalated
0,CMPL0001,2024-12-17,quality_damage,"Poor quality material, not as shown in photos",5,False
1,CMPL0002,2025-02-20,wrong_product,"Ordered size M, received size L",4,False
2,CMPL0003,2025-03-23,size_fit,Size inconsistency across different products,6,False
3,CMPL0004,2024-11-11,refund_delay,"Money stuck for 3 weeks, very disappointed",2,False
4,CMPL0005,2024-10-04,wrong_product,Received used/worn product instead of new,3,False


In [21]:
# Check data types
print("Data Types:")
print(df.dtypes)
print("\n" + "="*60)

# Check for missing values
print("\nMissing Values:")
missing_summary = df.isnull().sum()
missing_pct = (missing_summary / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_summary,
    'Percentage': missing_pct
}).sort_values('Missing Count', ascending=False)
print(missing_df[missing_df['Missing Count'] > 0])

Data Types:
order_id                  object
order_date                object
category                  object
order_value                int64
city                      object
city_tier                 object
warehouse                 object
payment_method            object
dispatch_date             object
delivery_date             object
delivery_status           object
is_returned                 bool
return_initiated_date     object
return_reason             object
refund_completed_date     object
refund_status             object
refund_delay_days        float64
sla_breach                object
dtype: object


Missing Values:
                       Missing Count  Percentage
sla_breach                     34341      68.682
return_initiated_date          34341      68.682
refund_delay_days              34341      68.682
refund_status                  34341      68.682
refund_completed_date          34341      68.682
return_reason                  34341      68.682
delivery_date      

In [22]:
print("Basic Statistics:")
df.describe()

Basic Statistics:


,order_value,refund_delay_days
count,50000.000000,15659.000000
mean,1680.054500,6.309918
std,818.171358,7.155290
min,299.000000,0.000000
25%,1049.000000,2.000000
50%,1518.000000,4.000000
75%,2188.000000,7.000000
max,3999.000000,49.000000


In [23]:
# Convert date columns to datetime
date_columns = ['order_date', 'dispatch_date', 'delivery_date', 
                'return_initiated_date', 'refund_completed_date']

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Convert boolean column
df['is_returned'] = df['is_returned'].astype(bool)
df['sla_breach'] = df['sla_breach'].astype('boolean')  # Nullable boolean

print("Date conversions complete!")
print(df.dtypes)

Date conversions complete!
order_id                         object
order_date               datetime64[ns]
category                         object
order_value                       int64
city                             object
city_tier                        object
warehouse                        object
payment_method                   object
dispatch_date            datetime64[ns]
delivery_date            datetime64[ns]
delivery_status                  object
is_returned                        bool
return_initiated_date    datetime64[ns]
return_reason                    object
refund_completed_date    datetime64[ns]
refund_status                    object
refund_delay_days               float64
sla_breach                      boolean
dtype: object


In [24]:
# Calculate time-based features

# 1. Days from order to dispatch
df['order_to_dispatch_days'] = (df['dispatch_date'] - df['order_date']).dt.days

# 2. Days from dispatch to delivery
df['dispatch_to_delivery_days'] = (df['delivery_date'] - df['dispatch_date']).dt.days

# 3. Total delivery time (order to delivery)
df['total_delivery_days'] = (df['delivery_date'] - df['order_date']).dt.days

# 4. Days from delivery to return initiation
df['delivery_to_return_days'] = (df['return_initiated_date'] - df['delivery_date']).dt.days

print("Time-based features created!")

Time-based features created!


In [25]:
# Create categorical features

# 1. Order month (for seasonality analysis)
df['order_month'] = df['order_date'].dt.month
df['order_month_name'] = df['order_date'].dt.strftime('%B')

# 2. Order day of week
df['order_day_of_week'] = df['order_date'].dt.day_name()

# 3. Price bucket
df['price_bucket'] = pd.cut(df['order_value'], 
                             bins=[0, 1000, 2000, 3000, 10000],
                             labels=['Budget (<₹1K)', 'Mid (₹1-2K)', 
                                     'Premium (₹2-3K)', 'Luxury (>₹3K)'])

# 4. Return window (early vs late returns)
df['return_window'] = df['delivery_to_return_days'].apply(
    lambda x: 'Early (1-7 days)' if pd.notna(x) and x <= 7 
    else ('Late (8-15 days)' if pd.notna(x) and x > 7 else None)
)

print("Categorical features created!")
df[['order_month_name', 'price_bucket', 'return_window']].head()

Categorical features created!


,order_month_name,price_bucket,return_window
0,October,Mid (₹1-2K),None
1,October,Premium (₹2-3K),Early (1-7 days)
2,March,Mid (₹1-2K),None
3,November,Mid (₹1-2K),Early (1-7 days)
4,December,Mid (₹1-2K),None


In [26]:
# Binary flags

# 1. Is order from metro city?
df['is_metro'] = df['city_tier'] == 'Metro'

# 2. Is payment COD?
df['is_cod'] = df['payment_method'] == 'COD'

# 3. Supply chain failure flag
df['supply_chain_failure'] = ~df['delivery_status'].isin(['Delivered'])

# 4. High-value order (>₹2000)
df['high_value'] = df['order_value'] > 2000

print("Binary flags created!")

Binary flags created!


In [27]:
# Validation 1: Ensure dispatch date >= order date
invalid_dispatch = df[df['dispatch_to_delivery_days'] < 0]
print(f"Invalid dispatch-delivery sequences: {len(invalid_dispatch)}")

# Validation 2: Returns should only happen for delivered orders
invalid_returns = df[df['is_returned'] & (df['delivery_status'] != 'Delivered')]
print(f"Invalid returns (non-delivered orders): {len(invalid_returns)}")

# Validation 3: Return reason should exist for all returned orders
missing_reason = df[df['is_returned'] & df['return_reason'].isna()]
print(f"Returned orders missing reason: {len(missing_reason)}")

print("\nValidation complete - all checks passed!" if len(invalid_dispatch) == 0 
      and len(invalid_returns) == 0 and len(missing_reason) == 0 else "⚠️ Issues found!")

Invalid dispatch-delivery sequences: 0
Invalid returns (non-delivered orders): 0
Returned orders missing reason: 0

Validation complete - all checks passed!


In [28]:
# Overall return rate
overall_return_rate = df['is_returned'].mean() * 100
print(f"Overall Return Rate: {overall_return_rate:.2f}%")

# Return rate by category
print("\nReturn Rate by Category:")
category_returns = df.groupby('category')['is_returned'].agg(['sum', 'count', 'mean'])
category_returns['return_rate_%'] = category_returns['mean'] * 100
category_returns = category_returns.sort_values('return_rate_%', ascending=False)
print(category_returns)

Overall Return Rate: 31.32%

Return Rate by Category:
              sum  count      mean  return_rate_%
category                                         
Dresses      4164   7510  0.554461      55.446072
Bottoms      3265  10070  0.324230      32.423039
Tops         5021  17417  0.288282      28.828156
Footwear     2464   9081  0.271336      27.133576
Accessories   745   5922  0.125802      12.580209


In [29]:
# Return rate by city tier
print("Return Rate by City Tier:")
tier_returns = df.groupby('city_tier')['is_returned'].agg(['sum', 'count', 'mean'])
tier_returns['return_rate_%'] = tier_returns['mean'] * 100
tier_returns = tier_returns.sort_values('return_rate_%', ascending=False)
print(tier_returns)

Return Rate by City Tier:
            sum  count      mean  return_rate_%
city_tier                                      
Metro      9274  27425  0.338159      33.815861
Tier2      4416  15065  0.293130      29.312977
Tier3      1969   7510  0.262184      26.218375


In [30]:
# Return reasons distribution
print("\nTop Return Reasons:")
return_reasons = df[df['is_returned']]['return_reason'].value_counts()
return_reasons_pct = (return_reasons / return_reasons.sum()) * 100
reason_df = pd.DataFrame({
    'Count': return_reasons,
    'Percentage': return_reasons_pct
})
print(reason_df)


Top Return Reasons:
                       Count  Percentage
return_reason                           
Size/Fit Issue          8249   52.678971
Quality/Damage          2869   18.321732
Wrong Product Shipped   2350   15.007344
Changed Mind            1607   10.262469
Color Mismatch           584    3.729485


In [31]:
# Refund SLA breach analysis
returns_df = df[df['is_returned']].copy()
sla_breach_rate = returns_df['sla_breach'].sum() / len(returns_df) * 100
print(f"\nRefund SLA Breach Rate: {sla_breach_rate:.2f}%")
print(f"Total breaches: {returns_df['sla_breach'].sum()} out of {len(returns_df)} returns")


Refund SLA Breach Rate: 24.08%
Total breaches: 3771 out of 15659 returns


In [32]:
# Supply chain failure rate
supply_chain_failure_rate = df['supply_chain_failure'].mean() * 100
print(f"Supply Chain Failure Rate: {supply_chain_failure_rate:.2f}%")
print("\nFailure Types:")
print(df[df['supply_chain_failure']]['delivery_status'].value_counts())

Supply Chain Failure Rate: 4.91%

Failure Types:
delivery_status
Damaged in Transit    852
RTO                   816
Lost in Transit       786
Name: count, dtype: int64


In [33]:
# Save full cleaned dataset
df.to_csv('../data/processed/orders_cleaned.csv', index=False)
print(f"✅ Saved cleaned orders dataset: {len(df)} rows")

# Save returns-only dataset for detailed analysis
df_returns = df[df['is_returned']].copy()
df_returns.to_csv('../data/processed/returns_only.csv', index=False)
print(f"✅ Saved returns-only dataset: {len(df_returns)} rows")

# Save delivered orders (non-returned) for comparison
df_delivered = df[(df['delivery_status'] == 'Delivered') & (~df['is_returned'])].copy()
df_delivered.to_csv('../data/processed/delivered_no_returns.csv', index=False)
print(f"✅ Saved delivered (no returns) dataset: {len(df_delivered)} rows")

✅ Saved cleaned orders dataset: 50000 rows
✅ Saved returns-only dataset: 15659 rows
✅ Saved delivered (no returns) dataset: 31887 rows


In [34]:
print("="*60)
print("DATA CLEANING SUMMARY")
print("="*60)
print(f"\nTotal Orders: {len(df):,}")
print(f"Date Range: {df['order_date'].min().strftime('%Y-%m-%d')} to {df['order_date'].max().strftime('%Y-%m-%d')}")
print(f"\nDelivery Statistics:")
print(f"  - Successfully Delivered: {(df['delivery_status'] == 'Delivered').sum():,} ({(df['delivery_status'] == 'Delivered').mean()*100:.1f}%)")
print(f"  - Supply Chain Failures: {df['supply_chain_failure'].sum():,} ({df['supply_chain_failure'].mean()*100:.1f}%)")
print(f"\nReturns Statistics:")
print(f"  - Total Returns: {df['is_returned'].sum():,} ({overall_return_rate:.2f}%)")
print(f"  - SLA Breaches: {returns_df['sla_breach'].sum():,} ({sla_breach_rate:.2f}% of returns)")
print(f"\nCategory Distribution:")
for cat in df['category'].value_counts().index:
    count = df[df['category'] == cat].shape[0]
    pct = count / len(df) * 100
    print(f"  - {cat}: {count:,} ({pct:.1f}%)")
print("\n" + "="*60)
print("✅ Data cleaning complete! Ready for analysis.")
print("="*60)

DATA CLEANING SUMMARY

Total Orders: 50,000
Date Range: 2024-10-03 to 2025-04-01

Delivery Statistics:
  - Successfully Delivered: 47,546 (95.1%)
  - Supply Chain Failures: 2,454 (4.9%)

Returns Statistics:
  - Total Returns: 15,659 (31.32%)
  - SLA Breaches: 3,771 (24.08% of returns)

Category Distribution:
  - Tops: 17,417 (34.8%)
  - Bottoms: 10,070 (20.1%)
  - Footwear: 9,081 (18.2%)
  - Dresses: 7,510 (15.0%)
  - Accessories: 5,922 (11.8%)

✅ Data cleaning complete! Ready for analysis.
